# Exploratory Data Analysis (EDA)

**Objectif :** Comprendre les données pour définir une stratégie de modélisation optimale.

## Contenu
1. Chargement et aperçu des données
2. Analyse de la forme
3. Analyse des valeurs manquantes
4. Analyse de la target
5. Distribution des features
6. Relations features-target
7. Corrélations
8. Tests statistiques
9. Conclusions

## 1. Setup et Chargement des Données

In [ ]:
# Imports
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

# Configuration
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
# Charger les données
from src.data.preprocessing import load_data
from src.config import TARGET_FEATURE

df = load_data()
print(f"Dataset shape: {df.shape}")
df.head()

## 2. Analyse de la Forme

In [ ]:
# Informations générales
print(f"Nombre de lignes: {df.shape[0]:,}")
print(f"Nombre de colonnes: {df.shape[1]}")
print(f"\nMémoire utilisée: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Types de variables
print("Distribution des types de données:\n")
type_counts = df.dtypes.value_counts()
print(type_counts)

print(f"\nVariables numériques: {(df.dtypes == 'float64').sum() + (df.dtypes == 'int64').sum()}")
print(f"Variables catégorielles: {(df.dtypes == 'object').sum()}")

## 3. Analyse des Valeurs Manquantes

In [ ]:
# Calcul du taux de valeurs manquantes
missing_rate = (df.isna().sum() / len(df) * 100).sort_values(ascending=False)

print("Top 20 colonnes avec le plus de valeurs manquantes:\n")
print(missing_rate.head(20))

print(f"\nColonnes avec >90% de NaN: {(missing_rate > 90).sum()}")
print(f"Colonnes avec >50% de NaN: {(missing_rate > 50).sum()}")
print(f"Colonnes sans NaN: {(missing_rate == 0).sum()}")

In [ ]:
# Visualisation des valeurs manquantes
plt.figure(figsize=(20, 10))
sns.heatmap(df.isna(), cbar=False, cmap='viridis', yticklabels=False)
plt.title('Heatmap des Valeurs Manquantes (jaune = manquant)', fontsize=16)
plt.xlabel('Features')
plt.tight_layout()
plt.show()

In [ ]:
# Identifier les groupes de features par taux de NaN
missing_groups = pd.DataFrame({
    'Column': missing_rate.index,
    'Missing_Rate': missing_rate.values
})

print("Groupes identifiés par taux de valeurs manquantes:\n")
print(f"Groupe 1 (75-88% NaN) - Tests viraux: {((missing_rate > 75) & (missing_rate < 88)).sum()} colonnes")
print(f"Groupe 2 (88-90% NaN) - Analyses sanguines: {((missing_rate > 88) & (missing_rate < 90)).sum()} colonnes")
print(f"Groupe 3 (>90% NaN) - Analyses spécifiques: {(missing_rate > 90).sum()} colonnes")

## 4. Analyse de la Target

In [ ]:
# Distribution de la target
target_counts = df[TARGET_FEATURE].value_counts()
target_pct = df[TARGET_FEATURE].value_counts(normalize=True) * 100

print(f"Distribution de {TARGET_FEATURE}:\n")
for cat in target_counts.index:
    print(f"{cat}: {target_counts[cat]:,} ({target_pct[cat]:.1f}%)")

# Visualisation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Countplot
sns.countplot(data=df, x=TARGET_FEATURE, ax=ax1, palette='Set2')
ax1.set_title('Distribution de la Target', fontsize=14)
ax1.set_ylabel('Nombre')
for p in ax1.patches:
    height = p.get_height()
    ax1.text(p.get_x() + p.get_width()/2., height,
             f'{int(height):,}\n({height/len(df)*100:.1f}%)',
             ha='center', va='bottom')

# Pie chart
ax2.pie(target_counts, labels=target_counts.index, autopct='%1.1f%%',
        colors=sns.color_palette('Set2'), startangle=90)
ax2.set_title('Proportion de la Target', fontsize=14)

plt.tight_layout()
plt.show()

print("\n⚠️ Dataset fortement déséquilibré : ~90% négatifs, ~10% positifs")

## 5. Sélection des Features Pertinentes

In [ ]:
# Sélectionner les features avec <90% de NaN
from src.data.preprocessing import select_features_by_missing_rate, get_feature_groups

df_selected = select_features_by_missing_rate(df)
blood_columns, viral_columns = get_feature_groups(df_selected)

print(f"Features sélectionnées: {df_selected.shape[1]} colonnes")
print(f"\nBlood features: {len(blood_columns)}")
print(blood_columns)
print(f"\nViral features: {len(viral_columns)}")
print(viral_columns)

## 6. Distribution des Features Numériques

In [ ]:
# Distributions des blood features
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(blood_columns[:16]):
    df_selected[col].hist(bins=30, ax=axes[idx], edgecolor='black', alpha=0.7)
    axes[idx].set_title(col, fontsize=10)
    axes[idx].set_xlabel('')

plt.suptitle('Distribution des Blood Features', fontsize=16, y=1.00)
plt.tight_layout()
plt.show()

print("📊 Les features sanguines sont standardisées et montrent des distributions variées")

## 7. Relations Features-Target

In [ ]:
# Créer des sous-ensembles par target
positive_df = df_selected[df_selected[TARGET_FEATURE] == 'positive']
negative_df = df_selected[df_selected[TARGET_FEATURE] == 'negative']

print(f"Cas positifs: {len(positive_df):,}")
print(f"Cas négatifs: {len(negative_df):,}")

In [ ]:
# Comparaison des distributions blood features par target
features_to_plot = ['Monocytes', 'Platelets', 'Leukocytes', 'Lymphocytes']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, col in enumerate(features_to_plot):
    if col in blood_columns:
        positive_df[col].dropna().hist(bins=30, alpha=0.5, label='Positive',
                                         ax=axes[idx], color='red', edgecolor='black')
        negative_df[col].dropna().hist(bins=30, alpha=0.5, label='Negative',
                                         ax=axes[idx], color='blue', edgecolor='black')
        axes[idx].set_title(f'Distribution de {col}', fontsize=12)
        axes[idx].set_xlabel('Valeur')
        axes[idx].set_ylabel('Fréquence')
        axes[idx].legend()

plt.suptitle('Comparaison Blood Features : Positifs vs Négatifs', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Relation target / age
plt.figure(figsize=(12, 6))
sns.countplot(data=df_selected, x='Patient age quantile', hue=TARGET_FEATURE, palette='Set2')
plt.title('Distribution par Age Quantile et Target', fontsize=14)
plt.xlabel('Patient Age Quantile')
plt.ylabel('Count')
plt.legend(title='COVID-19')
plt.tight_layout()
plt.show()

## 8. Analyse des Corrélations

In [ ]:
# Matrice de corrélation des blood features
plt.figure(figsize=(14, 12))
corr_matrix = df_selected[blood_columns].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matrice de Corrélation - Blood Features', fontsize=16)
plt.tight_layout()
plt.show()

# Identifier les corrélations fortes
high_corr = (corr_matrix.abs() > 0.8) & (corr_matrix.abs() < 1.0)
print("\nPaires avec corrélation > 0.8:")
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if high_corr.iloc[i, j]:
            print(f"{corr_matrix.columns[i]} <-> {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

In [ ]:
# Corrélation avec l'âge
age_corr = df_selected.select_dtypes(include=['number']).corr()['Patient age quantile'].sort_values()
print("Corrélation avec Patient Age Quantile:\n")
print(age_corr)
print("\n📊 Très faible corrélation entre l'âge et les taux sanguins")

## 9. Tests Statistiques

In [ ]:
# Test t de Student pour comparer les moyennes
# H0: Les moyennes sont égales entre positifs et négatifs

alpha = 0.02
significant_features = []

print("Test t de Student (H0: moyennes égales)\n")
print(f"{'Feature':<50} {'p-value':<12} {'Résultat'}")
print("-" * 75)

for col in blood_columns:
    pos_data = positive_df[col].dropna()
    neg_data = negative_df[col].dropna()
    
    if len(pos_data) > 5 and len(neg_data) > 5:
        stat, p_value = ttest_ind(pos_data, neg_data)
        result = "H0 REJETÉE ✓" if p_value < alpha else "H0 acceptée"
        
        if p_value < alpha:
            significant_features.append(col)
        
        print(f"{col:<50} {p_value:<12.6f} {result}")

print(f"\n🔬 Features significativement différentes (p < {alpha}): {len(significant_features)}")
print(significant_features)

## 10. Analyse des Données Virales

In [ ]:
# Distribution des tests viraux
viral_summary = {}

for col in viral_columns[:5]:  # Top 5 tests viraux
    counts = df_selected[col].value_counts()
    viral_summary[col] = counts

print("Distribution des tests viraux:\n")
for col, counts in viral_summary.items():
    print(f"\n{col}:")
    print(counts)

## 11. Conclusions de l'EDA

### Points clés identifiés:

1. **Dataset:** 
   - 5,644 patients, 111 features
   - Fortement déséquilibré: 90% négatifs, 10% positifs

2. **Valeurs manquantes:**
   - 2 groupes principaux: tests viraux (76% NaN) et analyses sanguines (89% NaN)
   - Sélection nécessaire des features pertinentes

3. **Features importantes:**
   - Monocytes, Platelets, Leukocytes montrent des différences significatives
   - Âge peu corrélé aux taux sanguins
   - Tests viraux complémentaires utiles

4. **Stratégie de preprocessing:**
   - Sélectionner features avec <90% NaN
   - Feature engineering: créer variable "est malade" à partir des tests viraux
   - Gérer le déséquilibre des classes (métriques: F1, Recall)

### Prochaine étape:
→ **Preprocessing & Modeling** dans le notebook `02_Preprocessing_Modeling.ipynb`